# 강아지 주요 색상 추출 FastAPI 서버

이 노트북은 Kaggle Notebook에서 바로 실행할 수 있도록 작성했습니다.

- 강아지 이미지 업로드
- `rembg`로 배경 제거
- 강아지 픽셀만 분석
- `black`, `white`, `gray`, `brown` 네 가지 색상 중 비율이 높은 순서대로 반환
- FastAPI 서버로 `/predict` API 제공

Kaggle에서는 외부에서 Notebook 내부 서버에 바로 접속하기 어려울 수 있습니다. 이 노트북은 서버 실행과 Notebook 내부 테스트를 모두 포함합니다.

In [ ]:
# Kaggle Notebook에서 필요한 패키지 설치
# - fastapi: 이미지 업로드 API 서버
# - uvicorn: FastAPI 실행 서버
# - python-multipart: 파일 업로드 처리
# - rembg: 이미지 배경 제거
# - onnxruntime: rembg 모델 실행 엔진
# - pillow/opencv-python-headless/numpy: 이미지 처리
# - nest-asyncio: Jupyter Notebook 안에서 서버 실행 보조
# - requests: Notebook 내부 API 테스트
#
# 참고: Kaggle에서 rembg 모델과 패키지를 내려받으려면 Notebook Internet 옵션을 켜 주세요.

!pip install -q fastapi uvicorn python-multipart rembg onnxruntime pillow opencv-python-headless numpy nest-asyncio requests

In [ ]:
import io
import threading
from pathlib import Path

import cv2
import nest_asyncio
import numpy as np
import uvicorn
from fastapi import FastAPI, File, HTTPException, UploadFile
from fastapi.responses import JSONResponse, Response
from PIL import Image
from rembg import remove

# Jupyter/Kaggle Notebook 이벤트 루프 안에서 uvicorn을 실행하기 위한 설정
nest_asyncio.apply()

## 색상 추출 로직

`rembg`가 만든 투명 배경 이미지를 기준으로 알파 값이 충분히 높은 픽셀만 강아지 픽셀로 사용합니다. 이후 RGB/HSV 조건을 함께 사용해서 검정, 흰색, 회색, 갈색으로 분류합니다.

In [ ]:
COLOR_LABELS = ["black", "white", "gray", "brown"]


def load_image_from_bytes(image_bytes: bytes) -> Image.Image:
    """업로드된 이미지 바이트를 PIL RGB 이미지로 변환합니다."""
    try:
        image = Image.open(io.BytesIO(image_bytes)).convert("RGB")
    except Exception as exc:
        raise ValueError("이미지를 열 수 없습니다. jpg, jpeg, png 파일을 사용해 주세요.") from exc
    return image


def remove_background(image: Image.Image) -> Image.Image:
    """rembg를 이용해 배경을 제거하고 RGBA 이미지로 반환합니다."""
    output = remove(image)
    return output.convert("RGBA")


def get_dog_pixels(rgba_image: Image.Image, alpha_threshold: int = 40) -> np.ndarray:
    """투명하지 않은 영역만 강아지 픽셀로 간주해 RGB 배열로 반환합니다."""
    rgba = np.array(rgba_image)
    rgb = rgba[:, :, :3]
    alpha = rgba[:, :, 3]

    # 알파 값이 낮은 픽셀은 배경 또는 경계 노이즈로 보고 제외합니다.
    mask = alpha > alpha_threshold
    pixels = rgb[mask]

    if len(pixels) == 0:
        raise ValueError("배경 제거 후 분석할 강아지 픽셀이 없습니다.")

    return pixels


def classify_pixels_to_four_colors(pixels: np.ndarray) -> dict:
    """RGB 픽셀을 black, white, gray, brown 네 가지 색상으로 분류합니다."""
    pixels = pixels.astype(np.uint8)
    r = pixels[:, 0].astype(np.int16)
    g = pixels[:, 1].astype(np.int16)
    b = pixels[:, 2].astype(np.int16)

    # OpenCV HSV 변환은 RGB가 아니라 BGR 입력을 기대하므로 RGB -> HSV로 직접 변환합니다.
    hsv = cv2.cvtColor(pixels.reshape(-1, 1, 3), cv2.COLOR_RGB2HSV).reshape(-1, 3)
    h = hsv[:, 0].astype(np.int16)  # 0~179
    s = hsv[:, 1].astype(np.int16)  # 0~255
    v = hsv[:, 2].astype(np.int16)  # 0~255

    channel_range = np.maximum.reduce([r, g, b]) - np.minimum.reduce([r, g, b])

    # 검정: 밝기가 매우 낮은 픽셀
    black_mask = v < 65

    # 흰색: 밝고 채도가 낮은 픽셀
    white_mask = (v >= 190) & (s < 45)

    # 회색: 채도가 낮고 흰색/검정이 아닌 픽셀
    gray_mask = (s < 55) & (channel_range < 45) & ~black_mask & ~white_mask

    # 갈색: 빨강/노랑 계열 색상, 어느 정도 채도가 있고 너무 어둡지 않은 픽셀
    # OpenCV의 hue는 0~179 범위라서 대략 5~35가 갈색/주황/노랑 계열입니다.
    brown_mask = (
        (h >= 5) & (h <= 35) &
        (s >= 35) &
        (v >= 45) &
        (r >= g - 15) &
        (g >= b - 10) &
        ~black_mask & ~white_mask & ~gray_mask
    )

    # 위 조건에서 빠진 애매한 픽셀은 네 가지 색상 중 가장 가까운 기준색으로 보정합니다.
    assigned_mask = black_mask | white_mask | gray_mask | brown_mask
    unassigned_pixels = pixels[~assigned_mask].astype(np.float32)

    counts = {
        "black": int(black_mask.sum()),
        "white": int(white_mask.sum()),
        "gray": int(gray_mask.sum()),
        "brown": int(brown_mask.sum()),
    }

    if len(unassigned_pixels) > 0:
        # 기준색은 강아지 털색에서 흔한 대표값으로 설정했습니다.
        prototypes = {
            "black": np.array([20, 20, 20], dtype=np.float32),
            "white": np.array([235, 235, 225], dtype=np.float32),
            "gray": np.array([130, 130, 130], dtype=np.float32),
            "brown": np.array([130, 75, 35], dtype=np.float32),
        }

        labels = list(prototypes.keys())
        centers = np.stack([prototypes[label] for label in labels], axis=0)
        distances = np.linalg.norm(unassigned_pixels[:, None, :] - centers[None, :, :], axis=2)
        nearest = np.argmin(distances, axis=1)

        for index in nearest:
            counts[labels[int(index)]] += 1

    total = sum(counts.values())
    if total == 0:
        raise ValueError("분류 가능한 픽셀이 없습니다.")

    # 비율이 높은 순서대로 정렬합니다.
    result = []
    for color, count in sorted(counts.items(), key=lambda item: item[1], reverse=True):
        result.append({
            "color": color,
            "ratio": round(count / total, 4),
            "percentage": round(count / total * 100, 2),
            "pixel_count": count,
        })

    return {
        "total_pixels": total,
        "colors": result,
        "dominant_color": result[0]["color"],
    }


def analyze_dog_color(image_bytes: bytes) -> dict:
    """이미지 바이트를 받아 배경 제거 후 강아지 주요 색상 비율을 반환합니다."""
    image = load_image_from_bytes(image_bytes)
    dog_rgba = remove_background(image)
    dog_pixels = get_dog_pixels(dog_rgba)
    analysis = classify_pixels_to_four_colors(dog_pixels)
    analysis["image_size"] = {"width": image.width, "height": image.height}
    return analysis

## FastAPI 서버 코드

`POST /predict`에 이미지 파일을 `file` 필드로 업로드하면 색상 분석 결과를 JSON으로 반환합니다.

In [ ]:
app = FastAPI(title="Dog Color Picker API", version="1.0.0")


@app.get("/")
def root():
    """서버 상태 확인용 기본 엔드포인트입니다."""
    return {
        "message": "Dog Color Picker API is running.",
        "usage": "POST /predict with multipart form-data field named 'file'.",
        "colors": COLOR_LABELS,
    }


@app.post("/predict")
async def predict(file: UploadFile = File(...)):
    """업로드된 강아지 이미지에서 주요 털색 비율을 추출합니다."""
    if file.content_type not in {"image/jpeg", "image/png", "image/jpg", "image/webp"}:
        raise HTTPException(status_code=400, detail="jpg, png, webp 이미지 파일만 업로드해 주세요.")

    image_bytes = await file.read()

    try:
        result = analyze_dog_color(image_bytes)
    except ValueError as exc:
        raise HTTPException(status_code=400, detail=str(exc)) from exc
    except Exception as exc:
        raise HTTPException(status_code=500, detail=f"색상 분석 중 오류가 발생했습니다: {exc}") from exc

    return JSONResponse(result)


@app.post("/extract")
async def extract_dog(file: UploadFile = File(...)):
    """배경을 제거한 강아지 이미지만 PNG로 반환합니다."""
    if file.content_type not in {"image/jpeg", "image/png", "image/jpg", "image/webp"}:
        raise HTTPException(status_code=400, detail="jpg, png, webp 이미지 파일만 업로드해 주세요.")

    image_bytes = await file.read()

    try:
        image = load_image_from_bytes(image_bytes)
        dog_rgba = remove_background(image)
        buffer = io.BytesIO()
        dog_rgba.save(buffer, format="PNG")
    except ValueError as exc:
        raise HTTPException(status_code=400, detail=str(exc)) from exc
    except Exception as exc:
        raise HTTPException(status_code=500, detail=f"배경 제거 중 오류가 발생했습니다: {exc}") from exc

    return Response(content=buffer.getvalue(), media_type="image/png")

In [ ]:
# Kaggle Notebook 내부에서 FastAPI 서버 실행
# 실행 후 Notebook 안에서는 http://127.0.0.1:8000 으로 요청할 수 있습니다.

def run_server():
    config = uvicorn.Config(app, host="0.0.0.0", port=8000, log_level="info")
    server = uvicorn.Server(config)
    server.run()


if "server_thread" not in globals() or not server_thread.is_alive():
    server_thread = threading.Thread(target=run_server, daemon=True)
    server_thread.start()
    print("FastAPI 서버가 실행되었습니다: http://127.0.0.1:8000")
else:
    print("FastAPI 서버가 이미 실행 중입니다: http://127.0.0.1:8000")

print("API 문서: http://127.0.0.1:8000/docs")

## Notebook 내부 테스트 방법

Kaggle 오른쪽 패널의 **Add data** 또는 업로드 기능으로 강아지 이미지 파일을 추가한 뒤, 아래 `TEST_IMAGE_PATH`에 이미지 경로를 넣고 실행하세요.

In [ ]:
# 예시 경로를 실제 Kaggle 이미지 경로로 바꿔 주세요.
# 예: /kaggle/input/my-dog-image/dog.jpg

TEST_IMAGE_PATH = "/kaggle/input/your-dog-image/dog.jpg"

if Path(TEST_IMAGE_PATH).exists():
    with open(TEST_IMAGE_PATH, "rb") as f:
        result = analyze_dog_color(f.read())
    result
else:
    print(f"테스트 이미지가 없습니다. TEST_IMAGE_PATH를 실제 파일 경로로 수정해 주세요: {TEST_IMAGE_PATH}")

In [ ]:
# FastAPI 서버에 직접 요청하는 테스트 코드입니다.
# 서버 실행 셀을 먼저 실행한 뒤 사용하세요.

import requests

if Path(TEST_IMAGE_PATH).exists():
    with open(TEST_IMAGE_PATH, "rb") as f:
        response = requests.post(
            "http://127.0.0.1:8000/predict",
            files={"file": (Path(TEST_IMAGE_PATH).name, f, "image/jpeg")},
            timeout=120,
        )
    print(response.status_code)
    print(response.json())
else:
    print("TEST_IMAGE_PATH를 실제 이미지 경로로 수정한 뒤 다시 실행해 주세요.")

## 반환 결과 예시

```json
{
  "total_pixels": 123456,
  "colors": [
    {"color": "brown", "ratio": 0.6123, "percentage": 61.23, "pixel_count": 75592},
    {"color": "white", "ratio": 0.2011, "percentage": 20.11, "pixel_count": 24827},
    {"color": "black", "ratio": 0.1505, "percentage": 15.05, "pixel_count": 18580},
    {"color": "gray", "ratio": 0.0361, "percentage": 3.61, "pixel_count": 4457}
  ],
  "dominant_color": "brown",
  "image_size": {"width": 640, "height": 480}
}
```

`colors`는 항상 비율이 높은 순서대로 정렬됩니다.